**Author:** Adebanji Oluwatimileyin Adelowo  
**GitHub:** [adebanjiadelowo](https://github.com/adebanjiadelowo)

<div align="center">
<h1>Enterprise NL2SQL Pipeline</h1>
<h2>Stage 1 — Table Selector</h2>
<p>Uses a lightweight LLM to identify which tables are needed before building the full SQL prompt.</p>
</div>

<hr>

## Overview

In an enterprise database with dozens of tables, passing every schema to an LLM on each query is both expensive and noisy. This notebook implements **Stage 1** of the two-stage NL2SQL pipeline: given a user question and a catalogue of short table descriptions, a lightweight model returns only the tables that are actually needed.

**Why this matters:**
- Smaller prompts for Stage 2 → lower cost and fewer hallucinations
- Table descriptions are ~10 tokens each; full schemas are ~100–200 tokens each
- A 10-table database with average 150-token schemas would cost ~1 500 tokens per query without filtering

The output of this stage (a JSON list of table names) feeds directly into Stage 2 (`02_SQL_Generator.ipynb`), which assembles the focused schema prompt.

## Setup

In [ ]:
import os
import json
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# gpt-4o-mini is the current cost-efficient model replacing the retired gpt-3.5-turbo
TABLE_SELECTOR_MODEL = "gpt-4o-mini"

## Model Helper

In [ ]:
def query_model(system_prompt: str, user_message: str, model: str = TABLE_SELECTOR_MODEL) -> str:
    """Send a single-turn request and return the assistant's text response."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ],
        temperature=0,
    )
    return response.choices[0].message.content

## Table Catalogue

The catalogue holds one short description per table — just enough for the model to infer relevance. The full `CREATE TABLE` statements (with sample rows and few-shot SQL) are assembled in Stage 2 only for the selected tables.

In [ ]:
catalogue = [
    {"table": "employees",           "definition": "Core HR record: employee ID, full name, job title, department ID, hire date, office ID, manager ID"},
    {"table": "salary",              "definition": "Annual compensation per employee per year: base salary, bonus amount, currency"},
    {"table": "studies",             "definition": "Educational background: employee ID, institution name, degree type, field of study, graduation year"},
    {"table": "departments",         "definition": "Department registry: department ID, name, manager employee ID, annual budget"},
    {"table": "projects",            "definition": "Company projects: project ID, name, start date, end date, budget, status"},
    {"table": "employee_projects",   "definition": "Project assignments: which employees work on which projects, their role, and hours logged"},
    {"table": "performance_reviews", "definition": "Annual performance scores per employee: review year, score (1-5), reviewer ID, comments"},
    {"table": "offices",             "definition": "Office locations: city, country, address, capacity (number of desks)"},
]

df_catalogue = pd.DataFrame(catalogue)
print(df_catalogue.to_string(index=False))

In [ ]:
# Build the plain-text block that goes into the prompt
text_tables = "\n".join(
    f"{row['table']}: {row['definition']}" for _, row in df_catalogue.iterrows()
)
print(text_tables)

## Prompt Template

The system prompt establishes the task constraints. The user message carries the filled template so the model treats the question as the user turn — consistent with chat-model best practices.

In [ ]:
SYSTEM_PROMPT = (
    "You are a database assistant. "
    "Given a list of database tables and their descriptions, identify which tables are required "
    "to answer the user's question. "
    "Respond ONLY with a valid JSON array of table name strings, e.g. [\"table_a\", \"table_b\"]. "
    "Do not include any explanation or markdown fencing."
)

USER_TEMPLATE = """\
### Available tables
{tables}

### Question
{question}

Which tables are needed? Reply with a JSON array of table names only.
"""

## Table Selection Helper

Wraps the prompt construction, model call, and JSON parsing into one function. It also validates that every returned name exists in the catalogue, guarding against hallucinated table names.

In [ ]:
KNOWN_TABLES = {row["table"] for row in catalogue}

def select_tables(question: str) -> list[str]:
    """Return the table names needed to answer *question*."""
    user_msg = USER_TEMPLATE.format(tables=text_tables, question=question)
    raw = query_model(SYSTEM_PROMPT, user_msg)

    selected = json.loads(raw)

    unknown = [t for t in selected if t not in KNOWN_TABLES]
    if unknown:
        raise ValueError(f"Model returned unknown table names: {unknown}")

    return selected

## Test Cases

Five questions that exercise different table combinations — from single-table lookups to four-table joins.

In [ ]:
# Single-domain: only compensation data needed
q1 = "Who is the highest-paid employee?"
print(f"Q: {q1}")
print(f"Tables: {select_tables(q1)}")

In [ ]:
# Cross-domain: links education to compensation
q2 = "Which educational institution has alumni with the highest average salary?"
print(f"Q: {q2}")
print(f"Tables: {select_tables(q2)}")

In [ ]:
# Project workload: no salary or education needed
q3 = "How many hours has each employee logged across all projects?"
print(f"Q: {q3}")
print(f"Tables: {select_tables(q3)}")

In [ ]:
# Department performance: links employees, departments, and reviews
q4 = "Which department has the highest average performance review score?"
print(f"Q: {q4}")
print(f"Tables: {select_tables(q4)}")

In [ ]:
# Office + project: multi-join across four tables
q5 = "Which office city has employees who worked the most hours on active projects?"
print(f"Q: {q5}")
print(f"Tables: {select_tables(q5)}")

## Summary Table

Run all questions and display results side-by-side to compare table selection coverage and token savings.

In [ ]:
test_questions = [
    ("Highest-paid employee",                               "Who is the highest-paid employee?"),
    ("Top-earning institution",                             "Which educational institution has alumni with the highest average salary?"),
    ("Hours logged per employee",                           "How many hours has each employee logged across all projects?"),
    ("Best-performing department",                          "Which department has the highest average performance review score?"),
    ("Most active office city by project hours",            "Which office city has employees who worked the most hours on active projects?"),
]

total_tables = len(catalogue)
rows = []
for label, question in test_questions:
    selected = select_tables(question)
    reduction = round((1 - len(selected) / total_tables) * 100)
    rows.append({
        "Question": label,
        "Selected tables": ", ".join(selected),
        "Count": len(selected),
        "Schema prompt reduction": f"{reduction}%",
    })

df_results = pd.DataFrame(rows)
df_results

## Conclusions

The table selector reliably narrows an 8-table catalogue down to only the relevant tables for each question:

- **Single-domain queries** (e.g. salary only) return 1–2 tables, cutting the schema prompt by ~75–85%.
- **Cross-domain queries** correctly pull in bridge tables (`employees`) that link domain-specific tables (`salary`, `studies`).
- **No hallucinated tables** were returned — the validation step enforces this at runtime.

The selected table list feeds directly into `02_SQL_Generator.ipynb`, where only those schemas (plus sample rows and few-shot examples) are assembled into the Stage 2 prompt.